# Notebook 08 — QA Testing: Small Sample Evaluation (10 Representative Examples)

Efficient QA validation using **10 diverse examples** + **all 6 model outputs**.

**Strategy**: 
- Select 10 representative examples (different genres, lengths, fact densities)
- Run inference on all 6 models (Exp1-3 LoRA/FFT)
- Display side-by-side for manual scoring
- ~30-60 min total manual review time

**Goal**: Validate that BERTScore improvement (0.68→0.84) = real semantic preservation, not metric noise.

**Integration with quantitative metrics**:
- Notebook 07 shows **BERTScore F1 improvement**: 0.68→0.84 from unidirectional training
- This notebook validates that improvement reflects **actual content preservation**
- Combined: metrics prove quality gain; QA proves it's meaningful

In [1]:
import sys
import json
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
TEST_JSONL = PROCESSED_DIR / 'test.jsonl'

# For inference (if needed)
sys.path.insert(0, str(ROOT))
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import AutoPeftModelForCausalLM
import torch

print('Setup complete.')
print(f'Test data path: {TEST_JSONL}')

Setup complete.
Test data path: /home/prasingh/data/Mormon-NLT/data/processed/test.jsonl


## 1. Select 10 Representative Examples (Diverse)

Sample strategically: different lengths, complexities, and content types

In [ ]:
# Select 10 diverse examples based on source text length (proxy for complexity)
with open(TEST_JSONL, encoding='utf-8') as f:
    test_data = [json.loads(line) for line in f if line.strip()]

# Get source lengths
source_lengths = []
for i, record in enumerate(test_data):
    msgs = record['messages']
    src = next(m['content'] for m in msgs if m['role'] == 'user')
    source_lengths.append((i, len(src)))

source_lengths.sort(key=lambda x: x[1])  # sort by length

# Sample 10: include short, medium, long examples
n_total = len(source_lengths)
sample_indices = [
    source_lengths[n_total // 10][0],      # Short (~10%)
    source_lengths[n_total // 5][0],       # Short-medium (~20%)
    source_lengths[n_total // 3][0],       # Medium (~33%)
    source_lengths[n_total // 2][0],       # Medium-long (~50%)
    source_lengths[2 * n_total // 3][0],   # Long (~67%)
    source_lengths[4 * n_total // 5][0],   # Long (~80%)
    source_lengths[9 * n_total // 10][0],  # Very long (~90%)
    source_lengths[n_total // 7][0],       # Additional diversity
    source_lengths[3 * n_total // 7][0],   # Additional diversity
    source_lengths[5 * n_total // 7][0],   # Additional diversity
]
sample_indices = sorted(set(sample_indices))[:10]  # Ensure unique, limit to 10

sample_data = [test_data[i] for i in sample_indices]

print(f'Selected {len(sample_data)} diverse examples:')
for idx, record in enumerate(sample_data):
    msgs = record['messages']
    src = next(m['content'] for m in msgs if m['role'] == 'user')
    print(f"  Example {idx+1}: {len(src)} chars — {src[:60]}...")

## 2. Load All 6 Models

In [ ]:
import gc

# Model paths
model_configs = {
    'Exp1 LoRA': {
        'adapter_path': ROOT / 'outputs' / 'exp1' / 'lora' / 'final_adapter',
        'is_lora': True,
    },
    'Exp2 LoRA': {
        'adapter_path': ROOT / 'outputs' / 'exp2' / 'lora' / 'final_adapter',
        'is_lora': True,
    },
    'Exp3 LoRA': {
        'adapter_path': ROOT / 'outputs' / 'exp3' / 'lora' / 'final_adapter',
        'is_lora': True,
    },
    'Exp1 FFT': {
        'model_path': ROOT / 'outputs' / 'exp1' / 'fft' / 'final_model',
        'is_lora': False,
    },
    'Exp2 FFT': {
        'model_path': ROOT / 'outputs' / 'exp2' / 'fft' / 'final_model',
        'is_lora': False,
    },
    'Exp3 FFT': {
        'model_path': ROOT / 'outputs' / 'exp3' / 'fft' / 'final_model',
        'is_lora': False,
    },
}

# Load models
models = {}
tokenizers = {}

print('Loading all 6 models...')
for variant, config in model_configs.items():
    try:
        if config['is_lora']:
            model = AutoPeftModelForCausalLM.from_pretrained(
                str(config['adapter_path']),
                torch_dtype=torch.bfloat16,
                device_map='auto',
                attn_implementation='sdpa',
            )
            tok = AutoTokenizer.from_pretrained(str(config['adapter_path']))
        else:
            model = AutoModelForCausalLM.from_pretrained(
                str(config['model_path']),
                torch_dtype=torch.bfloat16,
                device_map='auto',
                attn_implementation='sdpa',
            )
            tok = AutoTokenizer.from_pretrained(str(config['model_path']))
        
        model.eval()
        models[variant] = model
        tokenizers[variant] = tok
        print(f'  ✓ {variant}')
    except Exception as e:
        print(f'  ✗ {variant}: {str(e)[:60]}')

print(f'\nLoaded {len(models)}/6 models')

## 3. Run Inference on 10 Examples (All 6 Models)

In [ ]:
def generate_output(model, tokenizer, source_text, max_new_tokens=512):
    """Generate Shakespeare output from modern English input."""
    # Format as in training
    prompt = f"""You are an expert translator of Modern English into Shakespearean English. Translate the following modern English text into authentic Shakespearean style, preserving the meaning while using appropriate Early Modern English vocabulary, grammar, and poetic diction.

{source_text}"""
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated.strip()

# Run inference on 10 examples
qa_examples = []
print(f'Running inference on {len(sample_data)} examples across 6 models...')
print('(This may take 5-10 minutes)\n')

for ex_idx, record in enumerate(sample_data):
    msgs = record['messages']
    src = next(m['content'] for m in msgs if m['role'] == 'user')
    ref = next(m['content'] for m in msgs if m['role'] == 'assistant')
    
    example = {
        'id': ex_idx + 1,
        'source': src,
        'reference': ref,
        'outputs': {},
    }
    
    # Generate from all models
    for variant in ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']:
        if variant in models:
            try:
                output = generate_output(models[variant], tokenizers[variant], src)
                example['outputs'][variant] = output
                print(f"  Ex{ex_idx+1} × {variant:12} ✓")
            except Exception as e:
                example['outputs'][variant] = f'ERROR: {str(e)[:50]}'
                print(f"  Ex{ex_idx+1} × {variant:12} ✗")
        else:
            example['outputs'][variant] = 'Model not loaded'
    
    qa_examples.append(example)
    print()

print(f'Inference complete! Generated outputs for {len(qa_examples)} examples.')

## 4. Display Examples for Manual Scoring

For each example below:
1. Read the **Modern English source** (key info to preserve)
2. Compare all 6 model outputs
3. Score each: **✓** (preserves meaning) | **~** (partial) | **✗** (fails/inverts)

In [ ]:
def display_qa_example(example_dict):
    """Display a single QA example with all 6 model outputs."""
    ex_id = example_dict['id']
    source = example_dict['source']
    reference = example_dict['reference']
    outputs = example_dict['outputs']
    
    print('\n' + '='*120)
    print(f"EXAMPLE {ex_id}")
    print('='*120)
    
    print(f"\n[MODERN ENGLISH SOURCE]:")
    print(source)
    
    print(f"\n[REFERENCE SHAKESPEARE]:")
    print(reference)
    
    print(f"\n[MODEL OUTPUTS]:")
    print('-'*120)
    for variant in ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']:
        output = outputs.get(variant, 'N/A')
        print(f"\n{variant:12} → {output[:150]}..." if len(str(output)) > 150 else f"\n{variant:12} → {output}")
    
    print(f"\n[YOUR SCORES] (enter ✓, ~, or ✗ for each):")
    scores = {}
    for variant in ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']:
        # This is a template - user will manually enter scores
        scores[variant] = input(f"  {variant:12}: ")
    
    return scores

# Display all 10 examples
all_scores = {}
for i, example in enumerate(qa_examples):
    display_qa_example(example)
    all_scores[f"Example_{example['id']}"] = {
        'source': example['source'][:100],  # Truncate for storage
        'scores': {}  # Will be filled by user
    }
    
    # Prompt for scores
    print("\n" + "!"*120)
    print("ENTER YOUR SCORES NOW (✓/~/✗ for each model, space-separated):")
    score_input = input("Exp1LoRA Exp2LoRA Exp3LoRA Exp1FFT Exp2FFT Exp3FFT: ").split()
    
    if len(score_input) == 6:
        variants = ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']
        for variant, score in zip(variants, score_input):
            all_scores[f"Example_{example['id']}"]['scores'][variant] = score
    print("✓ Scores recorded.\n")

print("\n" + "="*120)
print("SCORING COMPLETE!")
print("="*120)

## 5. Aggregate QA Scores

Tally results from your manual scoring above:

# Tally scores
variants = ['Exp1 LoRA', 'Exp2 LoRA', 'Exp3 LoRA', 'Exp1 FFT', 'Exp2 FFT', 'Exp3 FFT']
qa_results = {v: {'success': 0, 'partial': 0, 'fail': 0} for v in variants}

# Count responses
for ex_key, ex_data in all_scores.items():
    for variant, score in ex_data['scores'].items():
        if score == '✓':
            qa_results[variant]['success'] += 1
        elif score == '~':
            qa_results[variant]['partial'] += 1
        elif score == '✗':
            qa_results[variant]['fail'] += 1

# Calculate preservation %
def calc_qa_score(counts, n=10):
    """Score = (✓ + 0.5×~) / n"""
    return (counts['success'] + 0.5 * counts['partial']) / n

print('\n' + '='*80)
print('QA PRESERVATION SCORES (10 Examples)')
print('='*80 + '\n')

results_sorted = sorted(
    [(v, calc_qa_score(qa_results[v])) for v in variants],
    key=lambda x: x[1],
    reverse=True
)

for variant, score in results_sorted:
    counts = qa_results[variant]
    print(f"{variant:12} → {score:.0%}  ({counts['success']}✓, {counts['partial']}~, {counts['fail']}✗)")

print()
print('Scoring: ✓ = Fully preserves meaning')
print('         ~ = Partially preserves (some context needed)')
print('         ✗ = Fails (meaning lost or inverted)')

# Save results
results_dir = ROOT / 'outputs' / 'exp3' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
with open(results_dir / 'qa_evaluation_10sample.json', 'w') as f:
    json.dump({
        'n_samples': 10,
        'qa_results': qa_results,
        'qa_scores': {v: calc_qa_score(qa_results[v]) for v in variants},
        'all_examples': all_scores,
    }, f, indent=2)

print(f'\n✓ Results saved to outputs/exp3/results/qa_evaluation_10sample.json')

## 6. Analysis: Validation Against BERTScore

Compare your QA results to projected BERTScore improvements: